In [35]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_stx"


In [36]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 44 firm-variable mappings across 29 firms.


firm_id,variable,notes,final_choice,num_candidates
AJAb.ST,COGS,"Candidate list was empty. Chose 'Property related operating expenses' from the Income Statement preview because it appears to be the most likely direct cost line tied to rental/property operations, whereas 'Property Costs' is unpopulated and may be a header. Manual review needed because this issuer is property/investment focused and the boundary between direct property costs and operating overhead is not fully clear from labels alone.",Income Statement::Property related operating expenses,0
AJAb.ST,XSGA_COMPONENTS,Used candidate 'Personnel expenses' and added 'Operating expenses' from the full preview because no explicit populated SG&A total row exists and 'Operating expenses' appears to be the closest additional overhead bucket. Excluded D&A-related SG&A row per instructions. Manual review needed because 'Operating expenses' may overlap partly with direct property costs depending on issuer-specific presentation. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Operating expenses,Income Statement::Personnel expenses; Income Statement::Operating expenses,2
BTSb.ST,XSGA_COMPONENTS,"Candidate list appears incomplete: no explicit SG&A/sales and administration row is provided. Selected the closest overhead component buckets from the full A-column preview. Manual review is needed because these broad operating expense lines may also include direct service-delivery costs in addition to SG&A. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Other external expenses, [Income Statement] Employee benefit expenses",Income Statement::Other external expenses; Income Statement::Employee benefit expenses,1
BUFAB.ST,XSGA_COMPONENTS,"Candidate list clearly misses the main SG&A component rows. Selected A-column labels outside the candidate list: 'Distribution costs' and 'Administative expenses' are the best SG&A components, with 'Other operating expenses' added as an overhead bucket. Excluded 'Depreciation/Amortization in SGA' per instructions. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Distribution costs, [Income Statement] Administative expenses",Income Statement::Distribution costs; Income Statement::Administative expenses; Income Statement::Other operating expenses,3
BULTEN.ST,BE,Candidate list misses the better owner-only equity line present in the balance sheet. Selected 'Equity attributable to Parent Company shareholders' to exclude non-controlling interests and avoid double counting with MIB. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Equity attributable to Parent Company shareholders,Balance Sheet::Equity attributable to Parent Company shareholders,2
BULTEN.ST,XSGA_COMPONENTS,"Used component rows and excluded subtotal 'Operating Expense'. Chose 'Selling expenses.' because the row without period appears to include separate depreciation. For administrative expense, candidate list lacks the cleaner ex-D&A variant distinction; selected 'Administrative expenses' from candidates, but this may include depreciation in some years because a separate depreciation breakout is shown below. Manual review recommended to confirm whether the intended SG&A should instead use the lower administrative component row.",Income Statement::Selling expenses.; Income Statement::Administrative expenses; Income Statement::Other operating expenses,5
CAMX.ST,XSGA_COMPONENTS,"Candidate list clearly misses 'Marketing and distribution costs', which is a core SG&A/overhead component present on the Income Statement. Included it from the full A-column labels along with candidate SG&A components. Manual review flagged because selection uses a row outside the candidate list. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Marketing and distribution costs",Income Statement::Marketing and distribution costs; Income Statement::Administrative expenses; Income State

In [37]:
df_review = pd.DataFrame(rows)

df_xsga = (
    df_review[df_review["variable"] == "XSGA_COMPONENTS"]
    .sort_values(["firm_id"])
    .reset_index(drop=True)
)

print(f"XSGA_COMPONENTS manual reviews: {len(df_xsga)} rows across {df_xsga['firm_id'].nunique() if len(df_xsga) else 0} firms.")
display(df_xsga)

XSGA_COMPONENTS manual reviews: 18 rows across 18 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,AJAb.ST,XSGA_COMPONENTS,Used candidate 'Personnel expenses' and added ...,Income Statement::Personnel expenses; Income S...,2
1,BTSb.ST,XSGA_COMPONENTS,Candidate list appears incomplete: no explicit...,Income Statement::Other external expenses; Inc...,1
2,BUFAB.ST,XSGA_COMPONENTS,Candidate list clearly misses the main SG&A co...,Income Statement::Distribution costs; Income S...,3
3,BULTEN.ST,XSGA_COMPONENTS,Used component rows and excluded subtotal 'Ope...,Income Statement::Selling expenses.; Income St...,5
4,CAMX.ST,XSGA_COMPONENTS,Candidate list clearly misses 'Marketing and d...,Income Statement::Marketing and distribution c...,3
5,CANTA.ST,XSGA_COMPONENTS,Candidate list appears to miss the most explic...,Income Statement::Administrative costs; Income...,2
6,CAST.ST,XSGA_COMPONENTS,Candidate list misses the populated overhead b...,Income Statement::Selling/General/Admin Expens...,4
7,CATb.ST,XSGA_COMPONENTS,No explicit SG&A subtotal is present. Used com...,Income Statement::Personnel costs; Income Stat...,2
8,CEVI.ST,XSGA_COMPONENTS,Candidate list only contains depreciation/amor...,Income Statement::Sales and marketing expenses...,3
9,CNCJOb.ST,XSGA_COMPONENTS,Candidate list was empty. Selected best overhe...,Income Statement::Selling Costs; Income Statem...,0
